In [1]:
import os, re, time
import pandas as pd
from tqdm.auto import tqdm
import plotly.express as px


CORPUS = "/Users/gmello/Documents/python_repos/mestrado/resultados_experimentos"
UA     = "/Users/gmello/Documents/python_repos/mestrado/ua-dd-saea"


ALGORITMOS = ['b1',   'b3',   'b4',   'b5m',  'b5r',
              'c122', 'c141', 'c149', 'c154', 'c217', 
              'c238', 'c262', 'e7',   'e74',  'e81',  'e103',
              # pisos
              'nsga2', 'nsga3', 
              'moead', 'moead_media',
              'smsemoa'
             ]

OFFLINES  = {'b5m': 'off', 'b5r': 'off', 'e103': 'off', 'moead_media': 'off'}


PROBLEMAS = ["MMF1", "MMF4", "MMF11_L", "MMF16_20",
             "ZDT1", "ZDT3", "ZDT4", "ZDT6", 
             "DTLZ1", "DTLZ2", "DTLZ3", "DTLZ4", "DTLZ7",
             "WFG1", "WFG2", "WFG4", "WFG5", "WFG9", 
             "BBOB_F1", "BBOB_F5", "BBOB_F17", "BBOB_F22", "BBOB_F37", "BBOB_F49", "BBOB_F55",
             "RE21", "ESTOQUE40", "DDMOP7"]


SEEDS = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,42]


TABELAS = {'real': 'real',           # avaliacoes reais realizadas
           'surrogate': 'surrogate', # avaliacoes surrogate + sonda realizadas
           'populacao': 'pop',       # individuos na populacao por geracao
           'tempo': 'timing',        # tempo de execucao
           'final': 'final'}         # avaliacao veradeira de solucoes finais em algoritmos offline

In [2]:
def ler_dataframe(algoritmo, problema, seed, tabela):
    try: return pd.read_parquet(f'{CORPUS}/{algoritmo}/{problema}/{seed}/exp_{OFFLINES.get(algoritmo, "main")}_{algoritmo}_{problema}_{seed}__{tabela}.parquet')
    except: return None
    

linhas = []
for algoritmo in tqdm(ALGORITMOS):
    for problema in PROBLEMAS:
        for seed in SEEDS:

            # leitura de dados
            dict_df = {k: ler_dataframe(algoritmo, problema, seed, tabela) for k, tabela in TABELAS.items()}
            info    = dict(algoritmo=algoritmo, problema=problema, seed=seed, orcamento=None, orcamento_gasto=None, geracoes_gasto=None, tempo_execucao=None)

            # determina orcamento do experimento + mede orcamento real gasto
            if dict_df['real'] is not None:
                D = dict_df['real'].filter(regex=r'^x\d+$').shape[1]
                info.update(orcamento = 31*D - 1)
                info.update(orcamento_gasto = len(dict_df['real']))

            # mede populacoes reais geradas
            if dict_df['populacao'] is not None:
                info['geracoes_gasto'] = dict_df['populacao'].geracao.max()

            # mede tempo total de execucao
            if dict_df['tempo'] is not None:
                df_tempo = dict_df['tempo']
                cols = ['tempo_geracao_s', 'tempo_pred_sonda_s', 'tempo_checkpoint_s'] if 'tempo_geracao_s' in df_tempo else ['tempo_fit_s', 'tempo_busca_s']
                info['tempo_execucao'] = df_tempo.reindex(columns=cols).sum().sum() / 3600

            # registra dados medidos
            for k in TABELAS:
                linhas.append({**info, 'output': k, 'existe': dict_df[k] is not None})

# unifica resultados
df = pd.DataFrame(linhas)

# determina status do experimento
df['geracoes_esperadas'] = df.groupby(['algoritmo', 'problema']).geracoes_gasto.transform('max')
completo = (df.orcamento_gasto >= df.orcamento) | (df.algoritmo.isin(OFFLINES) & (df.geracoes_gasto == df.geracoes_esperadas))
df['status'] = None
tem = df.orcamento_gasto.notna()
df.loc[tem, 'status'] = 'incompleto sem teto 12h'
df.loc[tem & (df.tempo_execucao >= 11*3600), 'status'] = 'teto 12h'
df.loc[completo, 'status'] = 'completo'
df['status'] = df['status'].fillna('nulo')

df

  0%|          | 0/21 [00:00<?, ?it/s]

,algoritmo,problema,seed,orcamento,orcamento_gasto,geracoes_gasto,tempo_execucao,output,existe,geracoes_esperadas,status
0,b1,MMF1,0,61.0,61.0,42.0,0.007720,real,True,72.0,completo
1,b1,MMF1,0,61.0,61.0,42.0,0.007720,surrogate,True,72.0,completo
2,b1,MMF1,0,61.0,61.0,42.0,0.007720,populacao,True,72.0,completo
3,b1,MMF1,0,61.0,61.0,42.0,0.007720,tempo,True,72.0,completo
4,b1,MMF1,0,61.0,61.0,42.0,0.007720,final,False,72.0,completo
...,...,...,...,...,...,...,...,...,...,...,...
88195,smsemoa,DDMOP7,42,526.0,526.0,18.0,0.591159,real,True,18.0,completo
88196,smsemoa,DDMOP7,42,526.0,526.0,18.0,0.591159,surrogate,True,18.0,completo
88197,smsemoa,DDMOP7,42,526.0,526.0,18.0,0.591159,populacao,True,18.0,completo
88198,smsemoa,DDMOP7,42,526.0,526.0,18.0,0.591159,tempo,True,18.0,completo


In [4]:
print(round(15047 / (21 * 28 * 30), 2))
df[df.output == 'real']['status'].value_counts()

0.85


status
completo                   15862
nulo                        1014
incompleto sem teto 12h      764
Name: count, dtype: int64

In [5]:
df[(df.output == 'real')&(df.status == 'incompleto sem teto 12h')].head()

,algoritmo,problema,seed,orcamento,orcamento_gasto,geracoes_gasto,tempo_execucao,output,existe,geracoes_esperadas,status
21575,c122,MMF16_20,25,619.0,564.0,345.0,4.685233,real,True,400.0,incompleto sem teto 12h
21635,c122,ZDT1,7,929.0,923.0,594.0,8.406286,real,True,600.0,incompleto sem teto 12h
21740,c122,ZDT1,28,929.0,913.0,584.0,11.918024,real,True,600.0,incompleto sem teto 12h
21745,c122,ZDT1,42,929.0,894.0,565.0,11.988909,real,True,600.0,incompleto sem teto 12h
21770,c122,ZDT3,4,929.0,892.0,563.0,11.959092,real,True,600.0,incompleto sem teto 12h


## 1. Análise completude experimentos

Conclusões
* [0] 14 algoritmos + 5 pisos e 27 problemas bem completos! \o/
* [1] estoque40 vira sub experimento (bem incompleto)
* [2] c154(JES) não rolou  --- deixar no cap4 e explicar no 5 que nao deu
    * tem varios incompletos dele -> explicar teto 12h e talvez sub estudo mostrar parciais 


In [6]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

real = df[df.output == 'real']
paineis = {
    '(1) todos os completos': real[real.status == 'completo'],
    '(2) completos limpeza':  real[(real.status == 'completo') & (real.algoritmo != 'c154') & (real.problema != 'ESTOQUE40')],
    '(3) incompletos':        real[real.status == 'incompleto sem teto 12h'],
    '(4) nulos':              real[real.status == 'nulo'],
}

fig = make_subplots(rows=2, cols=2, subplot_titles=list(paineis))
for i, sub in enumerate(paineis.values()):
    p = sub.pivot_table(index='algoritmo', columns='problema', values='seed', aggfunc='count')
    fig.add_trace(go.Heatmap(z=p.values, x=p.columns, y=p.index, coloraxis='coloraxis'), row=i//2 + 1, col=i%2 + 1)
fig.update_yaxes(autorange='reversed')
fig.update_layout(width=1300, height=1000)
fig.show()